# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AyeshaRiaz66/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
Rule:

Pages with high impressions, good CTR, high engagement and recent updates receive higher scores.

Reason Codes:
• LOW_CTR
• LOW_POSITION
• STALE_CONTENT
• HIGH_VALUE

Action Labels:
• Improve Title
• Improve SEO
• Refresh Content
• Monitor

In [15]:
import pandas as pd
import numpy as np

# Make a copy
baseline = df.copy()

# Rule score
baseline["score"] = (
    baseline["impressions_90d"]*0.35 +
    baseline["ctr"]*2000 +
    (100-baseline["avg_position"])*20 +
    baseline["engagement_rate"]*10 +
    baseline["word_count"]/100 -
    baseline["days_since_last_update"]*0.2
)

# Reason Code
def reason(row):
    if row["ctr"] < 2:
        return "LOW_CTR"
    elif row["avg_position"] > 10:
        return "LOW_POSITION"
    elif row["days_since_last_update"] > 180:
        return "STALE_CONTENT"
    else:
        return "HIGH_VALUE"

baseline["reason_code"] = baseline.apply(reason,axis=1)

# Action Label
def action(row):
    if row["reason_code"]=="LOW_CTR":
        return "Improve Title"
    elif row["reason_code"]=="LOW_POSITION":
        return "Improve SEO"
    elif row["reason_code"]=="STALE_CONTENT":
        return "Refresh Content"
    else:
        return "Monitor"

baseline["action"] = baseline.apply(action,axis=1)

print(baseline[["score","reason_code","action"]].head())

          score reason_code         action
10870  29702.16     LOW_CTR  Improve Title
22197  10054.56     LOW_CTR  Improve Title
16648  14359.62     LOW_CTR  Improve Title
18803  53798.90     LOW_CTR  Improve Title
20926  47490.92     LOW_CTR  Improve Title


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
The queue is ranked from highest score to lowest score.

The ranked queue is exported as baseline_action_score.csv.

In [16]:
baseline = baseline.sort_values(
    by="score",
    ascending=False
).reset_index(drop=True)

baseline["Rank"] = baseline.index+1

baseline.to_csv("baseline_action_score.csv",index=False)

print("CSV saved successfully.")

display(
baseline[
[
"Rank",
"content_id",
"score",
"reason_code",
"action"
]
].head(20))

CSV saved successfully.


,Rank,content_id,score,reason_code,action
0,1,content_006b16e7a2e7,203003.97,HIGH_VALUE,Monitor
1,2,content_4272d3a330a3,202868.80,HIGH_VALUE,Monitor
2,3,content_b1e4f7904d85,202004.26,HIGH_VALUE,Monitor
3,4,content_98458bafe297,201989.51,HIGH_VALUE,Monitor
4,5,content_bc2c0c7243df,201985.08,HIGH_VALUE,Monitor
5,6,content_bf398aa7400e,201964.51,HIGH_VALUE,Monitor
6,7,content_a84e013a5f94,201825.71,HIGH_VALUE,Monitor
7,8,content_cfa4d9f1bf0a,201528.47,LOW_POSITION,Improve SEO
8,9,content_6016b918a48f,201404.01,LOW_POSITION,Improve SEO
9,10,content_8c19996aa890,180630.45,LOW_CTR,Improve Title


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
The top 20 pages are manually reviewed.

For each page I recorded:
- Action
- Reason Code
- Confidence
- What could make the recommendation wrong

In [17]:
top20 = baseline.head(20)

for i,row in top20.iterrows():

    print("="*70)
    print("Rank:",row["Rank"])
    print("Content:",row["content_id"])
    print("Action:",row["action"])
    print("Reason:",row["reason_code"])

    if row["score"]>5000:
        confidence="High"
    elif row["score"]>2000:
        confidence="Medium"
    else:
        confidence="Low"

    print("Confidence:",confidence)

    print("What would make it wrong?")
    print("- Data may be outdated")
    print("- CTR may change later")
    print("- User behaviour may change")

Rank: 1
Content: content_006b16e7a2e7
Action: Monitor
Reason: HIGH_VALUE
Confidence: High
What would make it wrong?
- Data may be outdated
- CTR may change later
- User behaviour may change
Rank: 2
Content: content_4272d3a330a3
Action: Monitor
Reason: HIGH_VALUE
Confidence: High
What would make it wrong?
- Data may be outdated
- CTR may change later
- User behaviour may change
Rank: 3
Content: content_b1e4f7904d85
Action: Monitor
Reason: HIGH_VALUE
Confidence: High
What would make it wrong?
- Data may be outdated
- CTR may change later
- User behaviour may change
Rank: 4
Content: content_98458bafe297
Action: Monitor
Reason: HIGH_VALUE
Confidence: High
What would make it wrong?
- Data may be outdated
- CTR may change later
- User behaviour may change
Rank: 5
Content: content_bc2c0c7243df
Action: Monitor
Reason: HIGH_VALUE
Confidence: High
What would make it wrong?
- Data may be outdated
- CTR may change later
- User behaviour may change
Rank: 6
Content: content_bf398aa7400e
Action: Moni

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
Weak picks are reviewed to ensure no future information or label leakage was used.

The score only uses features available at prediction time.

In [18]:
print("Bottom 10 Pages")

display(
baseline.tail(10)[[
"content_id",
"score",
"reason_code",
"action"
]]
)

print()

print("Leakage Check")

print("No future columns were used.")

print("No labels were used in the score.")

print("Only current features were used:")
print("- impressions_90d")
print("- ctr")
print("- avg_position")
print("- engagement_rate")
print("- word_count")
print("- days_since_last_update")

Bottom 10 Pages


,content_id,score,reason_code,action
29990,content_7ba9b154acf6,NaN,LOW_CTR,Improve Title
29991,content_07ea0872b973,NaN,LOW_CTR,Improve Title
29992,content_77867ed726e1,NaN,LOW_CTR,Improve Title
29993,content_81a91fe32bc2,NaN,LOW_CTR,Improve Title
29994,content_fe5d259e6bc5,NaN,LOW_CTR,Improve Title
29995,content_6880eb215048,NaN,LOW_CTR,Improve Title
29996,content_677c622c0bfb,NaN,LOW_CTR,Improve Title
29997,content_e859812ce999,NaN,LOW_CTR,Improve Title
29998,content_9bd30342fd4a,NaN,LOW_CTR,Improve Title
29999,content_ab26273a7e7a,NaN,LOW_CTR,Improve Title



Leakage Check
No future columns were used.
No labels were used in the score.
Only current features were used:
- impressions_90d
- ctr
- avg_position
- engagement_rate
- word_count
- days_since_last_update


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.